In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import os


def clean_salary_job(val):
    if pd.isna(val):
        return np.nan
    
    val = str(val).strip()
    
    if val.isnumeric():
        return int(val)
    
    if val.endswith("L"):
        # Handle cases like "10.5L" or "10L"
        val = val.replace("L", "")
        if val.replace('.', '', 1).isdigit():
            return float(val) * 100000
    
    if val.endswith("k"):
        val = val.replace("k", "")
        if val.replace('.', '', 1).isdigit():
            return float(val) * 1000
    
    # Return original if no pattern matched
    try:
        return float(val)
    except:
        return np.nan


def scrape_city_data(city, headers, max_pages=11):
    """Scrape company data for a specific city"""
    url_for_city = f"https://www.ambitionbox.com/companies-in-{city}"
    dictionary = {
        "company_name": [],
        "rating": [],
        "bio": [],
        "salary": [],
        "job": [],
        "founded_in": []
    }
    
    for page in range(1, max_pages + 1):
        print(f"Scraping {city} - Page {page}")
        
        try:
            pages_url = f"{url_for_city}?page={page}"
            resp = requests.get(pages_url, headers=headers, timeout=10)
            resp.raise_for_status()  # Check for HTTP errors
            
            soup = BeautifulSoup(resp.content, 'html.parser')
            company_card_wrappers = soup.find_all("div", class_="companyCardWrapper")
            
            if not company_card_wrappers:
                print(f"No company cards found on page {page} for {city}")
                break
            
            for data in company_card_wrappers:
                # Extract company name
                company_name_elem = data.find("h2")
                company_name = company_name_elem.text.strip() if company_name_elem else np.nan
                dictionary["company_name"].append(company_name)
                
                # Extract rating
                rating_elem = data.find("div", class_="rating_text")
                rating = rating_elem.text.strip() if rating_elem else np.nan
                dictionary["rating"].append(rating)
                
                # Extract bio
                bio_elem = data.find("span", class_="companyCardWrapper__interLinking")
                bio = bio_elem.text.strip() if bio_elem else np.nan
                dictionary["bio"].append(bio)
                
                # Now scrape additional details from company overview page
                if company_name and company_name != np.nan:
                    company_slug = company_name.lower().replace(' ', '-').replace('.', '').replace(',', '')
                    overview_url = f"https://www.ambitionbox.com/overview/{company_slug}-overview"
                    
                    try:
                        time.sleep(1)  # Add delay to avoid rate limiting
                        resp1 = requests.get(overview_url, headers=headers, timeout=10)
                        
                        if resp1.status_code == 200:
                            new_soup = BeautifulSoup(resp1.content, 'html.parser')
                            
                            # Extract salary
                            salary = np.nan
                            a_tag_salary = new_soup.find("a", title=lambda x: x and "Salaries" in x)
                            if a_tag_salary:
                                salary_elem = a_tag_salary.find("div", class_="text-primary-text font-pn-600 text-xs")
                                if salary_elem:
                                    salary = salary_elem.text.strip()
                            dictionary["salary"].append(salary)
                            
                            # Extract job count
                            job = np.nan
                            a_tag_job = new_soup.find("a", title=lambda x: x and "Jobs" in x)
                            if a_tag_job:
                                job_elem = a_tag_job.find("div", class_="text-primary-text font-pn-600 text-xs")
                                if job_elem:
                                    job = job_elem.text.strip()
                            dictionary["job"].append(job)
                            
                            # Extract founded year
                            founded_year = np.nan
                            year_elements = new_soup.find_all("div", 
                                class_="inline whitespace-pre-wrap break-words text-primary-text text-sm font-pn-600 flex-[6] md:flex-[auto]")
                            if year_elements and len(year_elements) > 0:
                                founded_year = year_elements[0].text.strip()
                            dictionary["founded_in"].append(founded_year)
                            
                        else:
                            # If overview page fails, append NaN values
                            dictionary["salary"].append(np.nan)
                            dictionary["job"].append(np.nan)
                            dictionary["founded_in"].append(np.nan)
                            
                    except Exception as e:
                        print(f"Error scraping {company_name}: {e}")
                        dictionary["salary"].append(np.nan)
                        dictionary["job"].append(np.nan)
                        dictionary["founded_in"].append(np.nan)
                else:
                    dictionary["salary"].append(np.nan)
                    dictionary["job"].append(np.nan)
                    dictionary["founded_in"].append(np.nan)
            
            time.sleep(2)  # Delay between pages
        
        except requests.RequestException as e:
            print(f"Error accessing page {page} for {city}: {e}")
            break
    
    return pd.DataFrame(dictionary)


def main():
    cities = ["new-delhi", "mumbai", "pune", "bangalore", "chennai", "noida", "jaipur"]
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Cache-Control": "max-age=0",
    }
    
    # Create output directory
    output_dir = "scraped_data"
    os.makedirs(output_dir, exist_ok=True)
    
    # Step 1: Scrape data for each city
    for city in cities:
        print(f"\n{'='*50}")
        print(f"Scraping data for {city}")
        print(f"{'='*50}")
        
        try:
            df = scrape_city_data(city, headers, max_pages=11)
            
            if not df.empty:
                # Save raw data
                raw_filename = os.path.join(output_dir, f"{city}_raw.csv")
                df.to_csv(raw_filename, index=False)
                print(f"Saved raw data for {city} with {len(df)} records")
            else:
                print(f"No data scraped for {city}")
                
        except Exception as e:
            print(f"Failed to scrape {city}: {e}")
    
    # Step 2: Process each city's data
    print(f"\n{'='*50}")
    print("Processing scraped data")
    print(f"{'='*50}")
    
    for city in cities:
        filepath = os.path.join(output_dir, f"{city}_raw.csv")
        
        if os.path.exists(filepath):
            try:
                df = pd.read_csv(filepath)
                
                if not df.empty:
                    # Clean job and salary columns
                    df["job"] = df["job"].apply(clean_salary_job)
                    df["salary"] = df["salary"].apply(clean_salary_job)
                    
                    # Fill missing values with median (only if there are values)
                    if df["job"].notna().any():
                        median_job = df["job"].median()
                        df["job"].fillna(median_job, inplace=True)
                    
                    if df["salary"].notna().any():
                        median_salary = df["salary"].median()
                        df["salary"].fillna(median_salary, inplace=True)
                    
                    # Split bio column into field and other
                    df[["field", "other"]] = df["bio"].str.split("|", n=1, expand=True)
                    
                    # Clean field and other columns
                    df["field"] = df["field"].str.strip() if df["field"].notna().any() else df["field"]
                    df["other"] = df["other"].str.strip() if df["other"].notna().any() else df["other"]
                    
                    # Save processed data
                    processed_filename = os.path.join(output_dir, f"{city}_processed.csv")
                    df.to_csv(processed_filename, index=False)
                    
                    # Show missing values statistics
                    print(f"\n{city} -------------->")
                    print(f"Total records: {len(df)}")
                    print(f"Missing in job column: {df['job'].isnull().sum()}")
                    print(f"Missing in salary column: {df['salary'].isnull().sum()}")
                    print(f"Missing in founded_in column: {df['founded_in'].isnull().sum()}")
                    
            except Exception as e:
                print(f"Error processing {city}: {e}")
        else:
            print(f"No data file found for {city}")


if __name__ == "__main__":
    main()


Scraping data for new-delhi
Scraping new-delhi - Page 1
Scraping new-delhi - Page 2
Scraping new-delhi - Page 3
Scraping new-delhi - Page 4
Scraping new-delhi - Page 5
Scraping new-delhi - Page 6
Scraping new-delhi - Page 7
Scraping new-delhi - Page 8
Scraping new-delhi - Page 9
Scraping new-delhi - Page 10
Scraping new-delhi - Page 11
Saved raw data for new-delhi with 220 records

Scraping data for mumbai
Scraping mumbai - Page 1
Scraping mumbai - Page 2
Scraping mumbai - Page 3
Scraping mumbai - Page 4
Scraping mumbai - Page 5
Scraping mumbai - Page 6
Scraping mumbai - Page 7
Scraping mumbai - Page 8
Scraping mumbai - Page 9
Scraping mumbai - Page 10
Scraping mumbai - Page 11
Saved raw data for mumbai with 220 records

Scraping data for pune
Scraping pune - Page 1
Scraping pune - Page 2
Scraping pune - Page 3
Scraping pune - Page 4
Scraping pune - Page 5
Scraping pune - Page 6
Scraping pune - Page 7
Scraping pune - Page 8
Scraping pune - Page 9
Scraping pune - Page 10
Scraping pune 

C:\Users\dell\AppData\Local\Temp\ipykernel_4340\1696871481.py:199: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["job"].fillna(median_job, inplace=True)
C:\Users\dell\AppData\Local\Temp\ipykernel_4340\1696871481.py:203: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

In [2]:
import os
import pandas as pd

# Current working directory (Jupyter safe)
BASE_DIR = os.getcwd()

SCRAPED_DIR = os.path.join(BASE_DIR, "scraped_data")

cities = [
    "new-delhi",
    "mumbai",
    "pune",
    "bangalore",
    "chennai",
    "noida",
    "jaipur"
]

dataframes = []

for city in cities:
    file_path = os.path.join(SCRAPED_DIR, f"{city}_processed.csv")
    print("Reading:", file_path)
    df = pd.read_csv(file_path)
    dataframes.append(df)

combined_df = pd.concat(dataframes, ignore_index=True)

output_path = os.path.join(SCRAPED_DIR, "all_processed_combined.csv")
combined_df.to_csv(output_path, index=False)

print("✅ Combined file saved at:", output_path)


Reading: c:\Users\dell\Desktop\AMBITION PROJECT\scraped_data\new-delhi_processed.csv
Reading: c:\Users\dell\Desktop\AMBITION PROJECT\scraped_data\mumbai_processed.csv
Reading: c:\Users\dell\Desktop\AMBITION PROJECT\scraped_data\pune_processed.csv
Reading: c:\Users\dell\Desktop\AMBITION PROJECT\scraped_data\bangalore_processed.csv
Reading: c:\Users\dell\Desktop\AMBITION PROJECT\scraped_data\chennai_processed.csv
Reading: c:\Users\dell\Desktop\AMBITION PROJECT\scraped_data\noida_processed.csv
Reading: c:\Users\dell\Desktop\AMBITION PROJECT\scraped_data\jaipur_processed.csv
✅ Combined file saved at: c:\Users\dell\Desktop\AMBITION PROJECT\scraped_data\all_processed_combined.csv
